# Week 1 Day 2: the cheapest consistent agent, and how sure we can be

**The assignment:** using the agent, find the cheapest model that *consistently* gets at least 15 of the 25 claims right.

Most answers to this run a model once or three times and call 15+ consistent. But a model that scores 15, 16 and 15 might score 13 on its fourth run, and 3 runs out of 3 can't tell the difference. So this notebook asks the question behind the question: **how sure can we be that a model is consistent, and what does it cost to be sure?**

**Approach**
1. The lab's agent, unchanged, on any OpenRouter model. Each run gets its own record of decisions, so the claims run in parallel.
2. Screen tool-capable models with one run each, then confirm the ones within reach of the target with more runs.
3. Measure consistency two ways: the share of runs that reached the target, with a confidence bound, and the chance of reaching it estimated from how reliably the model gets each claim right.
4. A stability map of claims against models, to see *where* the inconsistency comes from.
5. A leaderboard of every cheap model measured in the lab's agent across the class, and a capability check of eight models from cheap to frontier, with the cost of each, to tell apart the misses a better model fixes from the ones only better context fixes.

**Two routes.** `FREE = True` uses `:free` models on the five tiny claims, with the target scaled to the same 60% (3 of 5). Free models allow about 50 requests a day and a claim takes about three, so every run is saved to a CSV beside the notebook and the study builds up over several days without repeating a run. `FREE = False` is the paid study on all 25 claims, for about $0.30.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from math import sqrt
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

## The lab's data, instructions and scoring

The same queue, the same instructions and the same check as the lab: the decision matches the handler's and the payout is within a dollar. On the free route, the queue is the five claims the repo marks as tiny.

In [ ]:
FREE = True
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


labels = {decision["claim_id"]: decision for decision in load("evals/decisions.json")}
claims = [claim for claim in load("claims.json") if not FREE or labels[claim["claim_id"]]["tiny"]]
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


print(f"{len(claims)} claims, {len(policies)} policies")

## The lab's agent, for any model

The tools are the lab's. Two changes: `make_tools` gives each run its own `recorded` dictionary, so several runs can't overwrite each other's decisions, and `MAX_TOKENS` caps each reply. Without the cap, some models ask OpenRouter to reserve up to 128,000 output tokens a call, and a low balance turns that into 402 errors that look like a weak model. A claim that errors or records nothing counts as wrong, as in the lab.

In [ ]:
TARGET = round(0.6 * len(claims))
WORKERS = 1 if FREE else 10
MAX_TOKENS = 4000


def make_tools(recorded):
    """The lab's two tools, recording into the dictionary given."""

    @tool
    def get_policy(policy_id: str) -> str:
        """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
        policy = policies.get(policy_id)
        return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
        """Record the decision on a claim in the claims system.

        payout is the amount to pay in US dollars, or 0 for a decline or a referral.
        rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
        """
        recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
        return f"Recorded {decision} on {claim_id}"

    return [get_policy, record_decision]


def make_agent(model, effort, recorded):
    """The lab's agent on any OpenRouter model."""
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                     reasoning_effort=effort, max_tokens=MAX_TOKENS, timeout=90, max_retries=3)
    return create_agent(model=llm, tools=make_tools(recorded), system_prompt=AGENT_SYSTEM_PROMPT)


def decide_with_agent(agent, recorded, claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    prompt = f"# Claim\n\n{json.dumps(claim, indent=2)}\n"
    cost, error = 0.0, ""
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": prompt}]}, {"recursion_limit": 12})
        cost = sum(message.response_metadata.get("token_usage", {}).get("cost") or 0.0
                   for message in result["messages"] if message.type == "ai")
    except Exception as e:
        error = f"{type(e).__name__}: {e}"[:200]
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    return {"claim_id": claim["claim_id"], **answer, "right": is_right(answer, labels[claim["claim_id"]]),
            "cost": cost, "seconds": round(time.time() - start, 1), "error": error}

`results` keeps every claim of every run, and is saved to a CSV after each run. `ensure_runs` tops a model up to a number of runs, so re-running a cell, or the notebook on another day, never pays twice for a run it already has. A run cut short by OpenRouter's limits (402 for credit, 429 for the rate or the daily cap) or by a dropped connection isn't the model's fault, so it is retried after a minute rather than scored.

In [ ]:
RESULTS_FILE = Path(f"how_sure_results_{'free' if FREE else 'paid'}.csv")
LIMIT_ERRORS = ("Error code: 402", "Error code: 429", "Connection error", "Timeout")

results = pd.read_csv(RESULTS_FILE, keep_default_na=False).to_dict("records") if RESULTS_FILE.exists() else []
print(f"{len(results)} claims already saved in {RESULTS_FILE}")


def run_queue(model, effort, run):
    """One run of the agent over the queue, with the claims in parallel."""
    recorded = {}
    agent = make_agent(model, effort, recorded)
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        rows = list(pool.map(lambda claim: decide_with_agent(agent, recorded, claim), claims))
    if any(code in row["error"] for row in rows for code in LIMIT_ERRORS):
        raise RuntimeError(f"{model} run {run} hit an OpenRouter limit or a network error, so it wasn't saved.")
    rows = [{"model": model, "effort": effort or "-", "run": run, **row} for row in rows]
    right, cost, errors = sum(r["right"] for r in rows), sum(r["cost"] for r in rows), sum(bool(r["error"]) for r in rows)
    print(f"{model} at {effort or '-'}, run {run}: {right} of {len(rows)} right for ${cost:.4f}, {errors} errors")
    return rows


def run_with_retries(model, effort, run, attempts=3, wait=60):
    """Run the queue, waiting a minute and trying again when an OpenRouter limit or a network error cuts a run short."""
    for attempt in range(1, attempts + 1):
        try:
            return run_queue(model, effort, run)
        except RuntimeError as e:
            print(f"{e} Attempt {attempt} of {attempts}.")
            if attempt < attempts:
                time.sleep(wait)
    return None


def ensure_runs(model, effort, runs):
    """Run a model until it has the number of runs given, saving after each run. Stop early at an OpenRouter limit."""
    done = {r["run"] for r in results if r["model"] == model}
    for run in range(1, runs + 1):
        if run not in done:
            rows = run_with_retries(model, effort, run)
            if rows is None:
                return
            results.extend(rows)
            pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)


def run_table():
    """One row for each run: the score, the cost and the errors."""
    df = pd.DataFrame(results)
    return df.groupby(["model", "effort", "run"], sort=False).agg(
        right=("right", "sum"), cost=("cost", "sum"), seconds=("seconds", "mean"), errors=("error", lambda e: (e != "").sum())).reset_index()

## The candidates

On the paid route, Luna at `low` is the lab's agent and the baseline, and the others are tool-capable models on OpenRouter that cost no more than Luna per token, from different labs. On the free route they are `:free` models that called a tool in a quick check; the free Gemma and Xing models were rate-limited by their providers when this ran. Models that can reason run at `low`, and the rest without a reasoning setting. The prices come live from OpenRouter, in dollars per million tokens.

In [ ]:
PAID_CANDIDATES = [
    "openai/gpt-6-luna",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "inclusionai/ling-3.0-flash",
    "qwen/qwen3.7-flash",
    "deepseek/deepseek-v4-flash",
    "mistralai/mistral-small-3.2-24b-instruct",
    "google/gemma-4-26b-a4b-it",
    "nvidia/nemotron-3.5-lightning",
    "z-ai/glm-4.7-flash",
]
FREE_CANDIDATES = [
    "nvidia/nemotron-3.5-lightning:free",
    "nvidia/nemotron-3.5-super-vl:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
]
CANDIDATES = FREE_CANDIDATES if FREE else PAID_CANDIDATES
BASELINE = CANDIDATES[0]

catalog = {m["id"]: m for m in requests.get(f"{OPENROUTER_BASE_URL}/models", timeout=30).json()["data"]}
efforts = {model: "low" if "reasoning" in catalog[model]["supported_parameters"] else None for model in CANDIDATES}

pd.DataFrame([{
    "model": model,
    "effort": efforts[model] or "-",
    "input": float(catalog[model]["pricing"]["prompt"]) * 1e6,
    "output": float(catalog[model]["pricing"]["completion"]) * 1e6,
} for model in CANDIDATES]).round(3)

## Screen: one run each

In [ ]:
for model in CANDIDATES:
    ensure_runs(model, efforts[model], 1)

screen = run_table()
screen.sort_values("cost")

## Confirm: more runs for every model within reach

A model a little under the target in its screening run could reach it on another, so the bar to go on is 13 of 25, or 2 of 5 on the free route. The baseline always goes on. The screening run counts as the first run.

In [ ]:
SCREEN_BAR = TARGET - max(1, len(claims) // 12)
RUNS = 3 if FREE else 6

screened = run_table().groupby("model")["right"].first()
finalists = [model for model in CANDIDATES if model == BASELINE or screened.get(model, 0) >= SCREEN_BAR]
print(f"Finalists: {', '.join(finalists)}")

for model in finalists:
    ensure_runs(model, efforts[model], RUNS)

## What "consistently" means

Two ways to measure it, from the same runs:

- **Runs that reached the target**, with a 90% Wilson lower bound. This is honest but blunt. Even 6 runs out of 6 only shows, with 90% confidence, that the model reaches the target at least 69% of the time. Three out of three shows just 53%.
- **The estimated chance of reaching the target.** Each claim has its own chance of being right, which we estimate from the runs as (right + 1) / (runs + 2), so a claim that was right every time still keeps a chance of failing. The score of a run is the sum of those claims, a Poisson binomial, and we add up the chance that it reaches the target. This uses every claim of every run, 150 on the paid route, rather than just a handful of pass-or-fail runs.

We call a model consistent when it reached the target on every run *and* its estimated chance of reaching it is at least 90%.

In [ ]:
Z_90 = 1.645
CONSISTENT = 0.9


def wilson_lower(passes, runs, z=Z_90):
    """The lower end of the Wilson interval for a pass rate."""
    if runs == 0:
        return 0.0
    rate = passes / runs
    centre = rate + z * z / (2 * runs)
    spread = z * sqrt(rate * (1 - rate) / runs + z * z / (4 * runs * runs))
    return (centre - spread) / (1 + z * z / runs)


def chance_of_at_least(chances, target):
    """The chance that independent claims with these chances of being right give at least target right."""
    totals = [1.0]
    for chance in chances:
        totals = [(totals[k] if k < len(totals) else 0.0) * (1 - chance) + (totals[k - 1] * chance if k > 0 else 0.0)
                  for k in range(len(totals) + 1)]
    return sum(totals[target:])


def claim_chances(model):
    """Each claim's chance of being right for a model, smoothed so that no claim is certain."""
    df = pd.DataFrame(results)
    df = df[df["model"] == model]
    return df.groupby("claim_id")["right"].agg(lambda right: (right.sum() + 1) / (len(right) + 2))


def reliability():
    """One row for each finalist: its scores, what it costs and how consistent it is. Fewer than RUNS runs can't be consistent."""
    runs = run_table()
    runs = runs[runs["model"].isin(finalists)]
    rows = []
    for model, group in runs.groupby("model", sort=False):
        passes = int((group["right"] >= TARGET).sum())
        chance = chance_of_at_least(claim_chances(model), TARGET)
        rows.append({
            "model": model,
            "effort": group["effort"].iloc[0],
            "scores": " ".join(str(s) for s in group["right"]),
            "worst": group["right"].min(),
            "mean": group["right"].mean(),
            "runs_at_target": f"{passes} / {len(group)}",
            "wilson_90_lower": wilson_lower(passes, len(group)),
            "chance_of_target": chance,
            "consistent": len(group) >= RUNS and passes == len(group) and chance >= CONSISTENT,
            "cost_per_run": group["cost"].mean(),
            "seconds_per_claim": group["seconds"].mean(),
            "errors": group["errors"].sum(),
        })
    return pd.DataFrame(rows).sort_values("cost_per_run").round(4)


table = reliability()
table

In [ ]:
consistent = table[table["consistent"]]
if consistent.empty:
    print("No model was consistent by this definition")
else:
    best = consistent.iloc[0]
    print(f"The cheapest consistent model is {best['model']} at {best['effort']}: "
          f"scores {best['scores']}, ${best['cost_per_run']:.4f} a run, "
          f"an estimated {best['chance_of_target']:.0%} chance of reaching {TARGET}")

## Cost against consistency

Each point is a finalist: its cost for a run of the queue against its estimated chance of reaching the target. The dashed line is the 90% bar, and the answer is the leftmost point above it. Free models all cost nothing, so on the free route the chart uses seconds per claim instead.

In [ ]:
x, x_label = ("seconds_per_claim", "Seconds per claim") if FREE else ("cost_per_run", "Cost of a run of the queue ($, log scale)")
fig = px.scatter(table, x=x, y="chance_of_target", text="model", log_x=not FREE,
                 hover_data=["scores", "runs_at_target", "wilson_90_lower"],
                 labels={x: x_label, "chance_of_target": f"Estimated chance of {TARGET} or more"},
                 title="Cost against consistency")
fig.add_hline(y=CONSISTENT, line_dash="dash")
fig.update_traces(textposition="top center")
fig.show()

## Where the inconsistency comes from

Each cell is the share of a finalist's runs that got the claim right. Most claims are always right or always wrong. The few in between are the flaky ones, and they decide whether a model scores 14 or 16.

In [ ]:
finals = pd.DataFrame(results)
finals = finals[finals["model"].isin(table["model"])]
stability = finals.pivot_table(index="claim_id", columns="model", values="right", aggfunc="mean")
stability["difficulty"] = pd.Categorical([labels[claim_id]["difficulty"] for claim_id in stability.index],
                                         categories=["routine", "hard", "manual"], ordered=True)
stability = stability.sort_values(["difficulty", "claim_id"])

fig = px.imshow(stability[table["model"]], color_continuous_scale="RdYlGn", zmin=0, zmax=1, aspect="auto",
                labels={"color": "share right"}, title="Share of runs right, by claim and model")
fig.update_yaxes(ticktext=[f"{c} ({d})" for c, d in zip(stability.index, stability["difficulty"])],
                 tickvals=list(stability.index))
fig.show()

In [ ]:
def classify(share):
    """Always right, always wrong or flaky."""
    return "always right" if share == 1 else "always wrong" if share == 0 else "flaky"


kinds = stability[table["model"]].map(classify)
summary = pd.DataFrame({model: kinds[model].value_counts() for model in table["model"]}).fillna(0).astype(int).T
summary

Here are the flaky claims for the cheapest consistent model, or for the baseline if none qualified, with what each claim is there to test.

In [ ]:
focus = consistent.iloc[0]["model"] if not consistent.empty else BASELINE
flaky = stability.index[(stability[focus] > 0) & (stability[focus] < 1)]

print(f"{len(flaky)} flaky claims for {focus}")
pd.DataFrame([{
    "claim_id": claim_id,
    "difficulty": labels[claim_id]["difficulty"],
    "share_right": round(stability.loc[claim_id, focus], 2),
    "handler": f"{labels[claim_id]['decision']} ${labels[claim_id]['payout']:,.2f}",
    "tests": labels[claim_id]["tests"],
} for claim_id in flaky], columns=["claim_id", "difficulty", "share_right", "handler", "tests"])

And the whole spend on this notebook, screening and confirming together.

In [ ]:
runs = run_table()
print(f"{len(runs)} runs of the queue, {len(results):,} claims, ${runs['cost'].sum():.4f} in all")

## The cheap-model leaderboard: every measured run of the lab's agent

Each run of the lab's agent on the 25 claims is independent evidence, whoever ran it. This pools every paid run of the lab's agent with a measured cost published in `community_contributions` with the first paid screen of this notebook, and scores each model on consistency the same way. Where a contribution reported the worst and the mean of three runs, that is enough to count the runs that reached 15. The cost is the measured OpenRouter cost of a run, averaged over the sources.

In [ ]:
QUEUE_TARGET = 15

# (source, scores, runs that reached 15, number of runs, measured cost of a run)
MEASURED_RUNS = {
    "openai/gpt-6-luna (low)": [("published", "17, 18, 18, 17", 4, 4, 0.0071), ("this notebook", "17", 1, 1, 0.0107)],
    "inclusionai/ling-3.0-flash": [("published", "17, 14, 16", 2, 3, 0.0060), ("this notebook", "15", 1, 1, 0.0062)],
    "qwen/qwen3.7-flash (low)": [("published", "worst 15, mean 15.7", 3, 3, 0.0180), ("this notebook", "15", 1, 1, 0.0178)],
    "deepseek/deepseek-v4-flash": [("published", "16", 1, 1, 0.0124)],
    "deepseek/deepseek-v4-flash-0731 (low)": [("published", "worst 15, mean 16.0", 3, 3, 0.0236)],
    "deepseek/deepseek-v4.1-flash (low)": [("published", "worst 17, mean 17.7", 3, 3, 0.0316)],
    "nvidia/nemotron-3.5-lightning (low)": [("published", "16, 16, 16", 3, 3, 0.0353)],
    "openai/gpt-oss-120b (low)": [("this notebook", "11", 0, 1, 0.0100)],
    "openai/gpt-oss-20b (low)": [("this notebook", "11", 0, 1, 0.0087)],
}

leaderboard = []
for model, sources in MEASURED_RUNS.items():
    passes = sum(source[2] for source in sources)
    runs = sum(source[3] for source in sources)
    costs = [source[4] for source in sources if source[4] is not None]
    leaderboard.append({
        "model": model,
        "scores (source)": "; ".join(f"{scores} ({source})" for source, scores, *_ in sources),
        "runs_at_15": f"{passes} / {runs}",
        "wilson_90_lower": round(wilson_lower(passes, runs), 2),
        "cost_per_run": round(sum(costs) / len(costs), 4),
    })

leaderboard = pd.DataFrame(leaderboard).sort_values("cost_per_run")
leaderboard

The cheapest model is not the answer if it can't be trusted. This chart puts each model's cost against the evidence that it reaches 15: the higher a point, the more runs back it up. Ling is the cheapest, but one of its four runs missed. Luna has the most evidence of any model, for barely more.

In [ ]:
fig = px.scatter(leaderboard, x="cost_per_run", y="wilson_90_lower", text="model", log_x=True,
                 hover_data=["scores (source)", "runs_at_15"],
                 labels={"cost_per_run": "Measured cost of a run of 25 claims ($, log scale)",
                         "wilson_90_lower": "90% lower bound on the chance of reaching 15"},
                 title="Cheap models in the lab's agent: cost against the evidence of consistency")
fig.update_traces(textposition="top center")
fig.show()

## A capability check across eight models

To tell capability misses from context misses, eight models from cheap to frontier each got the same single prompt: the lab's instructions, the policy wording and the 25 claims with their policy records, with none of the handlers' decisions. Each one returned a decision, payout and rationale for every claim, saved in `capability_decisions.csv`. They're scored here with the lab's `is_right`.

Each model has one run. It sees all 25 claims in one context and works out payouts itself, without the lab's tools, so this check isolates what the model knows from what the harness does.

The cost of a run is the lab's prompt for each claim (the system prompt with the policy wording, then the policy record and the claim) at 4 characters a token, with 300 output tokens a claim, at OpenRouter's live list prices. For Luna this gives $0.0105, against the $0.0107 the lab's agent actually cost on Luna in this notebook's paid screen. MAI Code Flash isn't on OpenRouter, so it has no price.

In [ ]:
CHECK_MODELS = {
    "grok-4.6": "x-ai/grok-4.6",
    "gemini-3.8-flash": "google/gemini-3.8-flash",
    "claude-sonnet-5.5": "anthropic/claude-sonnet-5.5",
    "claude-haiku-4.5": "anthropic/claude-haiku-4.5",
    "gpt-5.6-luna": "openai/gpt-5.6-luna",
    "gpt-6-luna": "openai/gpt-6-luna",
    "claude-haiku-5.5": "anthropic/claude-haiku-5.5",
    "mai-code-1.1-flash": None,
}
CHARS_PER_TOKEN = 4
OUTPUT_TOKENS_PER_CLAIM = 300

queue = load("claims.json")
input_tokens = sum(len(AGENT_SYSTEM_PROMPT) + len(json.dumps(policies[claim["policy_id"]], indent=2)) + len(json.dumps(claim, indent=2))
                   for claim in queue) / CHARS_PER_TOKEN
output_tokens = OUTPUT_TOKENS_PER_CLAIM * len(queue)


def list_price_cost(model_id):
    """The cost of one run of the queue at OpenRouter's list prices, or NaN for a model OpenRouter doesn't list."""
    if model_id is None:
        return float("nan")
    pricing = catalog[model_id]["pricing"]
    return input_tokens * float(pricing["prompt"]) + output_tokens * float(pricing["completion"])


capability = pd.read_csv("capability_decisions.csv")
capability["right"] = [is_right(row, labels[row["claim_id"]]) for row in capability.to_dict("records")]

scores = capability.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")[["routine", "hard", "manual"]]
scores.insert(0, "right", scores.sum(axis=1))
scores["cost_per_run"] = [list_price_cost(CHECK_MODELS[model]) for model in scores.index]
scores["times_luna"] = scores["cost_per_run"] / scores.loc["gpt-6-luna", "cost_per_run"]

print(f"About {input_tokens:,.0f} input and {output_tokens:,} output tokens for a run of {len(queue)} claims")
scores.sort_values(["right", "cost_per_run"], ascending=[False, True]).round(4)

Here are the claims that every model missed, and how each model did on CLM-1021.

In [ ]:
missed = capability.groupby("claim_id")["right"].sum()
print("Missed by every model:", ", ".join(f"{claim_id} ({labels[claim_id]['difficulty']})" for claim_id in missed[missed == 0].index))

capability[capability["claim_id"] == "CLM-1021"][["model", "decision", "payout", "right", "rationale"]]

## Key takeaways

**Answer to the assignment:** the cheapest model with strong evidence that it reaches 15 of 25 consistently is the lab's own Luna at `low`. It reached 15 on 5 runs out of 5, at about $0.009 a run of the queue. Ling 3.0 Flash costs a little less but missed on one run in four. Qwen 3.7 Flash is also consistent (4 out of 4), at twice the cost.

The evidence comes from three places: a leaderboard of every cheap model measured in the lab's agent (runs published in `community_contributions` plus this notebook's paid screen), a capability check of eight models given all 25 claims in a single prompt, and this notebook's free runs (three free Nemotron models on the five tiny claims, with a target of 3 of 5, the same 60% as 15 of 25).

| cheap models in the lab's agent | scores out of 25 (source) | runs at 15+ | 90% Wilson lower bound | cost a run |
|---|---|---|---|---|
| Ling 3.0 Flash | 17, 14, 16 (published); 15 (this notebook) | 3 / 4 | 0.36 | $0.0061 |
| gpt-oss-20b | 11 (this notebook) | 0 / 1 | 0.00 | $0.0087 |
| **GPT-6 Luna (low)** | **17, 18, 18, 17 (published); 17 (this notebook)** | **5 / 5** | **0.65** | **$0.0089** |
| gpt-oss-120b | 11 (this notebook) | 0 / 1 | 0.00 | $0.0100 |
| DeepSeek V4 Flash | 16 (published) | 1 / 1 | 0.27 | $0.0124 |
| Qwen 3.7 Flash (low) | worst 15, mean 15.7 (published); 15 (this notebook) | 4 / 4 | 0.60 | $0.0179 |
| DeepSeek V4 Flash 0731 (low) | worst 15, mean 16.0 (published) | 3 / 3 | 0.53 | $0.0236 |
| DeepSeek V4.1 Flash (low) | worst 17, mean 17.7 (published) | 3 / 3 | 0.53 | $0.0316 |
| Nemotron 3.5 Lightning (low) | 16, 16, 16 (published) | 3 / 3 | 0.53 | $0.0353 |

| capability check, one run each | right of 25 | routine (15) | hard (5) | manual (5) | cost a run (list price) | times Luna |
|---|---|---|---|---|---|---|
| Grok 4.6 | 20 | 15 | 5 | 0 | $0.181 | 17.2 |
| Gemini 3.8 Flash | 18 | 15 | 3 | 0 | $0.087 | 8.3 |
| GPT-6 Luna | 17 | 15 | 2 | 0 | $0.011 | 1.0 |
| Claude Sonnet 5.5 | 17 | 14 | 3 | 0 | $0.211 | 20.0 |
| Claude Haiku 5.5 | 16 | 13 | 3 | 0 | $0.011 | 1.0 |
| GPT-5.6 Luna | 15 | 15 | 0 | 0 | $0.023 | 2.1 |
| MAI Code 1.1 Flash | 12 | 12 | 0 | 0 | not on OpenRouter | |
| Claude Haiku 4.5 | 11 | 11 | 0 | 0 | $0.105 | 10.0 |

| free models, five tiny claims | scores of 5 | runs at 3+ | 90% Wilson lower bound | estimated chance of 3+ |
|---|---|---|---|---|
| nemotron-3.5-lightning:free | 3, 2, 3 | 2 / 3 | 0.25 | 0.55 |
| nemotron-3.5-super-vl:free | 3 | 1 / 1 | 0.27 | 0.57 |
| nemotron-3-ultra-550b-a55b:free | 3 | 1 / 1 | 0.27 | 0.57 |

1. **One run doesn't tell you much.** Lightning passed, failed, then passed again on the same five claims. Ling's best run beat Luna's worst, yet it fell to 14 on another. Even a perfect 3 out of 3 only shows, with 90% confidence, that a model passes at least 53% of the time. That's why the leaderboard ranks on evidence, not on a best score.
2. **Misses come in three kinds, and each needs a different fix.**
   - *Random misses.* CLM-1007, a $50 payout under a per-item limit, was approved twice and referred once by the same model. Twice Lightning also finished CLM-1021 without calling `record_decision` at all. More runs measure misses like these, and a harness that checks a decision was recorded would remove some of them.
   - *Capability misses.* On CLM-1021, every cheaper or older model (both Lunas, Haiku 4.5, MAI Code Flash and all three free Nemotrons, the 550B one included) paid $1,053 where the handler paid $825. They counted headphones and a game console as covered by Electronics Plus, which has a closed list of devices. Haiku 5.5, Sonnet, Gemini and Grok all paid the right $825. A stronger model fixes this miss, and so might a payout tool that applies the add-on's list in code.
   - *Context misses.* All eight models in the capability check, and every free model, missed all five `manual` claims. Their rules are only in the claims manual, so no model or number of runs will fix them. Only putting the manual in context will.
3. **Price is not quality.** Claude Haiku 4.5 costs 10 times as much as Luna and scored 11; Haiku 5.5 costs the same as Luna and scored 16. Sonnet costs 20 times Luna for the same 17. The free 550B Ultra scored the same as the small Super and was twice as slow. Model generation and fit to the task matter more than price or size.
4. **A stronger model buys the capability misses, at a price.** Grok reached 20, the most the policy wording allows, for $0.17 more than Luna a run of 25 claims: far less than a handler's time on one wrong decision. Repeat runs of Grok and Haiku 5.5 in the lab's agent are the next step.
5. **Pool the evidence.** Nine cheap models and every published run in one table say more than any single notebook. A shared results file that every student adds runs to would turn the course's PRs into one study with real statistical power.
6. **Plan for the limits.** More than half of this notebook's free runs were cut short by a rate limit or a dropped connection, a pattern other published runs show too. Scoring those as wrong makes a model look broken, as Gemma did in a first attempt at 0 of 5. So a run cut short this way is retried, never scored, and every finished run is saved.
